In [ ]:
from pathlib import Path
import arff
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


# 1. Data Loading

In [21]:
ROOT = Path.cwd()

if ROOT.name == 'notebooks':
    DATA_PATH = ROOT.parent / 'credit.arff'
else:
    DATA_PATH = 'credit.arff'
    
with open(DATA_PATH, 'r') as f:
    arff_data = arff.load(f)
    
data = arff_data['data']
column_names = [attr[0] for attr in arff_data['attributes']]
    
df = pd.DataFrame(data, columns=column_names)
df.head()

,Loan_ID,Loan_Amount_Requested,Length_Employed,Home_Owner,Annual_Income,Income_Verified,Purpose_Of_Loan,Debt_To_Income,Inquiries_Last_6Mo,Months_Since_Deliquency,Number_Open_Accounts,Total_Accounts,Gender,Interest_Rate
0,10139122,"35,000",3 years,NaN,160000.0,VERIFIED - income,credit_card,14.86,1,NaN,6,26,Male,None
1,10025461,"15,000",10+ years,Rent,41000.0,not verified,debt_consolidation,16.51,0,21.0,13,36,Female,None
2,10154747,"11,000",5 years,Mortgage,59000.0,not verified,debt_consolidation,21.75,0,NaN,11,20,Male,None
3,10032437,"12,000",NaN,Mortgage,72000.0,VERIFIED - income,debt_consolidation,15.73,1,NaN,7,20,Female,None
4,10060564,"20,000",< 1 year,Other,79404.0,VERIFIED - income,debt_consolidation,15.32,3,58.0,18,33,Female,None


In [22]:
df['Loan_Amount_Requested'] = df['Loan_Amount_Requested'].str.replace(',', '').astype(float)

In [23]:
X_reg = df.drop([
    'Annual_Income'
], axis=1).copy()
y_reg = df['Annual_Income'].copy()

y_reg = y_reg.fillna(y_reg.mean())

In [26]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

models = {
    'linear': LinearRegression(),
    'lasso': Lasso(alpha=1.0),
    'ridge': Ridge(alpha=0.01)
}

numeric_features = X_reg.select_dtypes(include='number').columns.tolist()
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = X_reg.select_dtypes(exclude='number').columns.tolist()
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

reg_preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

rows = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', reg_preprocessor),
        ('model', LinearRegression())
    ])
    pipe.fit(X_train_reg, y_train_reg)
    rows.append({'model': name, 'r2_test': r2_score(y_test_reg, pipe.predict(X_test_reg))})
    
pd.DataFrame(rows).sort_values('r2_test', ascending=False)


,model,r2_test
0,linear,0.260814
1,lasso,0.260814
2,ridge,0.260814


In our case of underfitting, Lasso and Ridge have no meaningfull effect, it could help if the model overfits